In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient

# 1. Configuración Pro
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 120

# 2. Conectar a MongoDB
client = MongoClient('mongodb://mongodb:27017')
db = client['steam_db']

# Cargar reseñas
df_rev = pd.DataFrame(list(db.reviews_raw.find()))

# --- MAPEO GEOGRÁFICO PARA EL MAPA DEL MUNDO ---
# Convertimos el idioma de Steam en un país para que PowerBI lo reconozca
mapa_paises = {
    'english': 'United States',
    'schinese': 'China',
    'russian': 'Russia',
    'spanish': 'Spain',
    'brazilian': 'Brazil',
    'german': 'Germany',
    'french': 'France',
    'japanese': 'Japan',
    'koreana': 'South Korea',
    'turkish': 'Turkey',
    'polish': 'Poland'
}
df_rev['pais_real'] = df_rev['language'].map(mapa_paises)

# --- FILTRADO POR VOLUMEN ---
# Solo analizamos idiomas con suficiente representación para que sea estadísticamente serio
top_idiomas = df_rev['language'].value_counts().head(8).index.tolist()
df_top = df_rev[df_rev['language'].isin(top_idiomas)].copy()

# =========================================================
# GRÁFICA 1: % DE QUEJAS POR CULTURA (Normalizado)
# =========================================================
palabras_neg = ['bug', 'crash', 'boring', 'expensive', 'short', 'hard']

# Marcamos con 1 si la palabra existe, 0 si no
for p in palabras_neg:
    df_top[f'is_{p}'] = df_top['review'].str.contains(p, case=False, na=False).astype(int)

# Calculamos la MEDIA (esto nos da el porcentaje de menciones por cada 100 reseñas)
# Filtramos solo por reseñas negativas para ver de qué se quejan cuando el juego no les gusta
df_neg = df_top[df_top['voted_up'] == False]
heatmap_neg = df_neg.groupby('language')[[f'is_{p}' for p in palabras_neg]].mean() * 100

plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_neg, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={'label': '% de menciones en críticas negativas'})
plt.title('Intensidad de Quejas: ¿Qué le molesta más a cada cultura? (%)', fontsize=14, pad=20)
plt.ylabel('Idioma / Región')
plt.xlabel('Motivo de la queja')
plt.show()

# =========================================================
# GRÁFICA 2: % DE VIRTUDES POR CULTURA (Normalizado)
# =========================================================
palabras_pos = ['fun', 'art', 'music', 'story', 'gameplay', 'relaxing']
for p in palabras_pos:
    df_top[f'is_{p}'] = df_top['review'].str.contains(p, case=False, na=False).astype(int)

# Filtramos solo por reseñas POSITIVAS
df_pos = df_top[df_top['voted_up'] == True]
heatmap_pos = df_pos.groupby('language')[[f'is_{p}' for p in palabras_pos]].mean() * 100

plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_pos, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={'label': '% de menciones en críticas positivas'})
plt.title('Puntos de Valor: ¿Qué apasiona a cada cultura? (%)', fontsize=14, pad=20)
plt.ylabel('Idioma / Región')
plt.xlabel('Virtud destacada')
plt.show()

# =========================================================
# 3. EXPORTACIÓN FINAL PARA POWERBI (EL MAPA REAL)
# =========================================================
# Guardamos el DataFrame con la columna 'pais_real' y los flags de quejas
# para que PowerBI pueda pintar el mapa del mundo
df_top.to_csv('/home/jovyan/work/steam_data_final.csv', index=False)

print(f"Análisis completado. Se ha generado 'steam_data_final.csv'.")
print(f"Descargar este archivo y usarlo en PowerBI con la columna 'pais_real'.")

KeyError: 'language'